# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic

# llama-cpp-python: PREBUILT CUDA 12.1 wheels — no compile, GPU-ready
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

!pip install -q transformers accelerate torch
print("✅ All dependencies installed")


# ── Cell 2: Configure ngrok tunnel ───────────────────────────────────────
from google.colab import userdata
from pyngrok import ngrok

try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    print('✅ Loaded NGROK_AUTHTOKEN from Colab secrets')
except Exception:
    NGROK_TOKEN = input('Enter ngrok authtoken (https://dashboard.ngrok.com): ').strip()

if not NGROK_TOKEN:
    raise ValueError('ngrok authtoken required')

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok configured")


In [ ]:
# ── Cell 3: Download LLaVA-1.6-Mistral-7B (best under 10 GB) ─────────────
# Main model:   ~4.4 GB  (cjpais/llava-1.6-mistral-7b-gguf)
# Vision enc:   ~631 MB  (mmproj-model-f16.gguf)
# Total:        ~5.0 GB  — fits on Colab T4 (15 GB VRAM)
import os

BASE = 'https://huggingface.co/cjpais/llava-1.6-mistral-7b-gguf/resolve/main'
MODEL_PATH  = '/content/llava-1.6-mistral-7b.Q4_K_M.gguf'
MMPROJ_PATH = '/content/mmproj-model-f16.gguf'

if not os.path.exists(MODEL_PATH):
    print('⬇️  Downloading main model (~4.4 GB)...')
    !wget -q --show-progress {BASE}/llava-1.6-mistral-7b.Q4_K_M.gguf -O {MODEL_PATH}
    print('✅ Main model done')
else:
    print(f'✅ Main model exists: {MODEL_PATH}')

if not os.path.exists(MMPROJ_PATH):
    print('⬇️  Downloading vision encoder mmproj (~631 MB)...')
    !wget -q --show-progress {BASE}/mmproj-model-f16.gguf -O {MMPROJ_PATH}
    print('✅ mmproj done')
else:
    print(f'✅ mmproj exists: {MMPROJ_PATH}')

total_gb = (os.path.getsize(MODEL_PATH) + os.path.getsize(MMPROJ_PATH)) / 1e9
print(f'\n📦 Total on disk: {total_gb:.2f} GB')


# ── Cell 4: Download server.py ────────────────────────────────────────────
!wget -q https://raw.githubusercontent.com/your-repo/visiontracker/main/remote_server/server.py -O /content/server.py
# Or upload manually via the Colab file panel
print("✅ server.py ready")


In [ ]:
# ── Cell 5: Start server + ngrok tunnel ──────────────────────────────────
import nest_asyncio, subprocess, threading, time
nest_asyncio.apply()

public_url = ngrok.connect(8000, 'http')

print('━' * 62)
print('🚀  VisionTracker Remote Server is READY')
print('━' * 62)
print(f'📡  Public URL : {public_url}')
print(f'    Health     : {public_url}/health')
print(f'    Identify   : {public_url}/identify')
print('━' * 62)
print('💡  Client command:')
print(f'    python main.py --remote-url {public_url}')
print('━' * 62)

def _run_server():
    cmd = [
        'python', '/content/server.py',
        '--model-path',  MODEL_PATH,
        '--mmproj-path', MMPROJ_PATH,
        '--host', '0.0.0.0',
        '--port', '8000',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')

threading.Thread(target=_run_server, daemon=True).start()
print('\nServer starting — keep this notebook running!')
try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    ngrok.disconnect(public_url)
    print('Stopped.')


## Cell 3: Choose and Download Model

Choose between GGUF or Safetensors format. Recommended: Gemma 3 4B IT.

**GGUF (Recommended for Colab T4):**
- Size: ~3.3GB (Q4_K_M quantized)
- VRAM: ~4GB
- Faster inference on T4

**Safetensors:**
- Size: ~8GB (full precision)
- VRAM: ~8-10GB
- Better quality but may OOM on T4

In [ ]:
import os

# Choose model format: 'gguf' or 'safetensors'
MODEL_FORMAT = "gguf"  # @param ["gguf", "safetensors"]

if MODEL_FORMAT == "gguf":
    # Gemma 3 4B IT GGUF
    MODEL_URL = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"
    MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"
    MODEL_PATH = f"/content/{MODEL_FILENAME}"
else:
    # For safetensors, we'll use a smaller vision model
    # Note: This requires more VRAM and may not fit on free T4
    MODEL_REPO = "google/gemma-3-4b-it"
    MODEL_PATH = f"/content/{MODEL_REPO.replace('/', '_')}"
    print(f"Safetensors model will be downloaded to: {MODEL_PATH}")

# Download GGUF if needed
if MODEL_FORMAT == "gguf" and not os.path.exists(MODEL_PATH):
    print(f"⬇️ Downloading {MODEL_FILENAME}...")
    print(f"   This is ~3.3GB, will take 2-3 minutes...")
    !wget -q --show-progress {MODEL_URL} -O {MODEL_PATH}
    print("✅ Download complete!")
elif MODEL_FORMAT == "gguf":
    print(f"✅ Model already exists at {MODEL_PATH}")

if MODEL_FORMAT == "gguf":
    print(f"📦 Model size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB")

## Cell 4: Download and Start Server

Downloads the server code and starts it with your chosen model.

In [ ]:
# Download the server code
!wget -q https://raw.githubusercontent.com/your-repo/visiontracker/main/remote_server/server.py -O /content/server.py

# For safetensors, download the model using huggingface
if MODEL_FORMAT == "safetensors":
    from transformers import AutoModelForVision2Seq, AutoProcessor
    print("⬇️ Downloading safetensors model (this may take a while)...")
    processor = AutoProcessor.from_pretrained(MODEL_REPO)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_REPO,
        torch_dtype="auto",
        device_map="auto"
    )
    # Save locally
    model.save_pretrained(MODEL_PATH)
    processor.save_pretrained(MODEL_PATH)
    print("✅ Model downloaded and saved!")
    # Clear memory
    del model, processor
    import torch
    torch.cuda.empty_cache()

print("✅ Server code ready!")

## Cell 5: Start ngrok Tunnel & Server

Starts the ngrok tunnel and the FastAPI server.

⚠️ **Important**: Keep this notebook running! If the runtime disconnects, the server stops.

### Output to copy:
- `Public URL` — use this in VisionTracker's `--remote-url` flag

In [ ]:
import nest_asyncio
import subprocess
import time

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Create tunnel
public_url = ngrok.connect(8000, "http")

print("━" * 60)
print("🚀 SERVER IS READY TO START!")
print("━" * 60)
print(f"📡 Public URL: {public_url}")
print(f"   Health: {public_url}/health")
print(f"   Identify: {public_url}/identify")
print("━" * 60)
print("\n💡 VisionTracker usage:")
print(f"   python main.py --remote-url {public_url}")
print("\n⚠️  Starting server now... Keep this notebook running!")
print("━" * 60)

# Start server in background
import threading

def start_server():
    cmd = ["python", "/content/server.py", "--model-path", MODEL_PATH, "--host", "0.0.0.0", "--port", "8000"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Keep alive
print("\nServer is running. Press Ctrl+C to stop.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping server...")
    ngrok.disconnect(public_url)

---

## 📝 Notes

### Keeping the Server Alive
- Colab free tier will disconnect after ~90 minutes of inactivity
- Keep the browser tab active
- Avoid closing your laptop/sleeping the machine

### ngrok Free Tier Limits
- 1 active tunnel at a time
- ~2 hour session limit (re-run cells to get new URL)
- 40 connections/minute rate limit (more than enough for VisionTracker)

### Troubleshooting

**Out of Memory errors:**
- Use GGUF format (smaller) instead of Safetensors
- Reduce batch size in VisionTracker: `--batch-size 4`
- Use CPU-only mode: set `n_gpu_layers=0` in server args

**Slow inference:**
- First request after loading is slow (CUDA warmup)
- Subsequent requests should be 1-3 seconds

**Connection refused:**
- Make sure this notebook is still running
- Check that the ngrok tunnel is active

### Alternative: Cloudflare Tunnel
If ngrok doesn't work, you can use Cloudflare tunnel:
```python
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'])
```